In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col, broadcast, collect_set, size # size add kiya

class Transformer():
    def __init__(self):
        pass
    
    def transform(self, inputDFs):
        pass

class AirpodsAfterIphoneTransformer(Transformer):
    def transform(self, inputDFs):
        ''' # Customers who have bought airpods after buying iphone '''
        transactionInputDF = inputDFs.get('transactionInputDF')
        customerInputDF = inputDFs.get("customerInputDF")
        
        windowSpec = Window.partitionBy("customer_id").orderBy("transaction_date")
        
        transformedDF = transactionInputDF.withColumn(
            "next_product_name", lead("product_name").over(windowSpec)
        )

        filteredDF = transformedDF.filter(
            (col("product_name") == 'iPhone') & (col("next_product_name") == 'AirPods')
        )

        joinDF = customerInputDF.join(broadcast(filteredDF), "customer_id")
        return joinDF # Return zaroori hai

class OnlyAirpodsAndIphone(Transformer):
    def transform(self, inputDFs):
        """
        Customer who have bought only iPhone and Airpods nothing else
        """
        # Extracting DataFrames
        transactionInputDF = inputDFs.get("transactionInputDF")
        customerInputDF = inputDFs.get("customerInputDF") # Isay yahan add kiya

        # Grouping logic
        groupedDF = transactionInputDF.groupBy("customer_id").agg(
            collect_set("product_name").alias("products")
        )

        # 4. Filter logic (Corrected Indentation and Syntax)
        from pyspark.sql.functions import array_contains # Array check ke liye best hai
        
        filteredDF = groupedDF.filter(
            (array_contains(col("products"), 'iPhone')) & 
            (array_contains(col("products"), 'AirPods')) & 
            (size(col("products")) == 2)
        )

        # 5. Customer Join Logic
        joinDF = customerInputDF.join(broadcast(filteredDF), "customer_id")

        print("JOINED DF (ONLY IPHONE & AIRPODS)")
        joinDF.select("customer_id", "customer_name", "location").show()
        return joinDF